In [5]:
import pandas as pd
import json
import random
from pathlib import Path

# ====================== PATHS ======================
JSON_FOLDER = '/content/drive/MyDrive/Sample/JSON_Files'
CSV_FOLDER = '/content/drive/MyDrive/Sample/CSV_upload'
OUTPUT_CLEAN_CSV = '/content/drive/MyDrive/Sample/Training_Data_Clean.csv'

print("Starting FINAL Part 2 - Clean Target Creation...\n")

score_map = {"Very Poor":1, "Poor":2, "Average":3, "Good":4, "Excellent":5}

all_data = []
json_files = list(Path(JSON_FOLDER).glob("*_Feedback.json"))

for json_path in json_files:
    base_name = json_path.stem.replace("_Feedback", "")
    csv_path = Path(CSV_FOLDER) / f"{base_name}_Recommendation.csv"

    print(f"Processing: {base_name}")

    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    student_df = pd.DataFrame(data)

    if csv_path.exists():
        book_summary = pd.read_csv(csv_path)
    else:
        book_summary = pd.DataFrame()

    # Convert to scores
    question_cols = [
        "How easy was this textbook to understand?",
        "How would you rate the conceptual clarity of the book?",
        "How engaging and interactive were the examples and exercises?",
        "How helpful were the visuals, diagrams, and illustrations?",
        "Overall, how satisfied are you with this textbook?",
        "Would you recommend this book to other students?",
        "How would you rate the value for money of this textbook?",
        "Was the book affordable given your financial situation?",
        "How useful was this book for scoring well in exams?",
        "How likely are you to use this book again in future courses?"
    ]

    for col in question_cols:
        if col in student_df.columns:
            student_df[col + '_score'] = student_df[col].map(score_map).fillna(3)

    student_df['Overall_Rating'] = student_df[[col + '_score' for col in question_cols]].mean(axis=1).round(2)

    # ================== CLEAN & BALANCED TARGET ==================
    # We now use multiple features instead of depending on one
    low_quality = (
        (student_df['Overall_Rating'] < 3.4) &
        (student_df['How would you rate the value for money of this textbook?_score'] <= 2) &
        (student_df['How easy was this textbook to understand?_score'] <= 3)
    )

    student_df['Needs_Replacement'] = low_quality.astype(int)

    # Force balance to ~30-35% Replace
    desired_ratio = 0.32
    desired_replace = int(len(student_df) * desired_ratio)
    current_replace = student_df['Needs_Replacement'].sum()

    if current_replace < desired_replace and current_replace > 0:
        replace_idx = student_df[student_df['Needs_Replacement'] == 1].index.tolist()
        extra = desired_replace - current_replace
        sampled = random.choices(replace_idx, k=extra)
        student_df.loc[sampled, 'Needs_Replacement'] = 1

    # Merge book summary
    if not book_summary.empty and 'Book_ID' in book_summary.columns:
        student_df = student_df.merge(book_summary[['Book_ID', 'avg_rating', 'num_reviews']],
                                      on='Book_ID', how='left')

    # Final columns
    final_cols = ['Student_ID', 'Book_ID', 'Book_Title', 'Author', 'Term',
                  'Overall_Rating', 'Needs_Replacement', 'Open_Ended_Comment']

    for col in question_cols:
        final_cols.append(col + '_score')

    if 'avg_rating' in student_df.columns:
        final_cols += ['avg_rating', 'num_reviews']

    clean_df = student_df[final_cols].copy()
    all_data.append(clean_df)

# Combine
training_df = pd.concat(all_data, ignore_index=True).reset_index(drop=True)

print(f"\n✅ Final Part 2 Completed!")
print(f"Total records: {len(training_df):,}")
print("\nTarget Distribution:")
print(training_df['Needs_Replacement'].value_counts())
print("\nPercentage:")
print(training_df['Needs_Replacement'].value_counts(normalize=True).round(4) * 100)

training_df.to_csv(OUTPUT_CLEAN_CSV, index=False, encoding='utf-8')
print(f"\nClean dataset saved at: {OUTPUT_CLEAN_CSV}")

Starting FINAL Part 2 - Clean Target Creation...

Processing: Student_Book_Interactions_ccis
Processing: Student_Book_Interactions_daemen
Processing: Student_Book_Interactions_betheltn
Processing: Student_Book_Interactions_eku
Processing: Student_Book_Interactions_campbell
Processing: Student_Book_Interactions_fiu
Processing: Student_Book_Interactions_niagara
Processing: Student_Book_Interactions_morgan
Processing: Student_Book_Interactions_glenville
Processing: Student_Book_Interactions_mercer
Processing: Student_Book_Interactions_onondaga
Processing: Student_Book_Interactions_oakwood
Processing: Student_Book_Interactions_nicholls

✅ Final Part 2 Completed!
Total records: 912,348

Target Distribution:
Needs_Replacement
0    722705
1    189643
Name: count, dtype: int64

Percentage:
Needs_Replacement
0    79.21
1    20.79
Name: proportion, dtype: float64

Clean dataset saved at: /content/drive/MyDrive/Sample/Training_Data_Clean.csv
